# Context

The use of drones in increasingly diverse fields (logistics, surveillance, infrastructure
inspection, or even rescue operations) comes with growing challenges of reliability and
safety.
 Drone failures can lead to loss of equipment, service interruptions, or risks to
people.
 It is therefore essential to prevent breakdowns through preventive maintenanceand
early fault detectionapproaches, in order to intervene before an incident occurs or to
quickly identify its cause.

# Objective

Based on the provided sensor data (in full or in part), the objective is to develop an
analysis and anomaly detection model capable of, depending on the chosen approach:</br>

 • Detecting the presence of a fault in the drone’s operation;</br>
 • Identifying the probable originof the fault;</br>
 • Assessing the severityof the detected fault to guide maintenance actions.</br>

 In short, the goal is to implement a predictive maintenanceapproach to improve the
availability, safety, and lifespan of drones.

# Loaing the data as watsonx

In [ ]:
from scipy.io import loadmat
import pandas as pd
import numpy as np
import re
import os

In [ ]:
def load_drone_data(path):
    data = loadmat(path)
    return data["commander_data"], data["QDrone_data"], data["stabilizer_data"]

In [ ]:
def extract_commander_data(commander):
    df = pd.DataFrame({
        "time": commander[0, :],
        "meas_x": commander[21, :],
        "meas_y": commander[22, :],
        "meas_z": commander[23, :],
        "meas_yaw": commander[24, :],
        "ref_x": commander[25, :],
        "ref_y": commander[26, :],
        "ref_z": commander[27, :],
        "ref_yaw": commander[28, :],
        "ref_thrust": commander[33, :],
        "ref_roll": commander[34, :],
        "ref_pitch": commander[35, :],
        "ref_yaw_rate": commander[36, :]
    })
    return df

In [ ]:
def extract_qdrone_data(qdrone):
    df = pd.DataFrame({
        "time": qdrone[0, :],
        # IMU 1 orientation
        "imu1_roll": qdrone[1, :],
        "imu1_pitch": qdrone[2, :],
        "imu1_yaw": qdrone[3, :],
        "imu1_roll_rate": qdrone[4, :],
        "imu1_pitch_rate": qdrone[5, :],
        "imu1_yaw_rate": qdrone[6, :],
        "imu1_roll_acc": qdrone[7, :],
        "imu1_pitch_acc": qdrone[8, :],
        "imu1_yaw_acc": qdrone[9, :],
        # IMU 2 orientation
        "imu2_roll": qdrone[10, :],
        "imu2_pitch": qdrone[11, :],
        "imu2_yaw": qdrone[12, :],
        "imu2_roll_rate": qdrone[13, :],
        "imu2_pitch_rate": qdrone[14, :],
        "imu2_yaw_rate": qdrone[15, :],
        "imu2_roll_acc": qdrone[16, :],
        "imu2_pitch_acc": qdrone[17, :],
        "imu2_yaw_acc": qdrone[18, :],
        # Other sensors
        "battery_level": qdrone[23, :],
        "gyro1_roll_rate": qdrone[26, :],
        "gyro1_pitch_rate": qdrone[27, :],
        "gyro1_yaw_rate": qdrone[28, :],
        "accel1_x": qdrone[29, :],
        "accel1_y": qdrone[30, :],
        "accel1_z": qdrone[31, :],
        "gyro2_roll_rate": qdrone[32, :],
        "gyro2_pitch_rate": qdrone[33, :],
        "gyro2_yaw_rate": qdrone[34, :],
        "accel2_x": qdrone[35, :],
        "accel2_y": qdrone[36, :],
        "accel2_z": qdrone[37, :],
        "height_range": qdrone[45, :],
        # Motor and ESC commands
        "front_left_motor_cmd": qdrone[46, :],
        "front_left_esc_cmd": qdrone[47, :],
        "front_right_motor_cmd": qdrone[48, :],
        "front_right_esc_cmd": qdrone[49, :],
        "back_left_motor_cmd": qdrone[50, :],
        "back_left_esc_cmd": qdrone[51, :],
        "back_right_motor_cmd": qdrone[52, :],
        "back_right_esc_cmd": qdrone[53, :]
    })
    return df

In [ ]:
def extract_stabilizer_data(stabilizer):
    df = pd.DataFrame({
        "time": stabilizer[0, :],
        "flight_mode": stabilizer[6, :]
    })
    return df

In [ ]:
def parse_filename(fname):
    match = re.match(r"F(\d+)_SV(\d+)_SP(\d+)_t(\d+)", fname)
    if match:
        F, SV, SP, t = map(int, match.groups())
        return F, SV, SP, t
    return None, None, None, None

In [ ]:
def process_file(file_path):
    fname = os.path.basename(file_path)
    F, SV, SP, t = parse_filename(fname)

    commander, qdrone, stabilizer = load_drone_data(file_path)

    df_commander = extract_commander_data(commander)
    df_qdrone = extract_qdrone_data(qdrone)
    df_stabilizer = extract_stabilizer_data(stabilizer)

    # Merge all on time (approx)
    df = df_commander.merge(df_qdrone, on="time", how="outer")
    df = df.merge(df_stabilizer, on="time", how="outer")

    # Add metadata columns
    df["fault_type"] = F
    df["severity"] = SV
    df["speed_level"] = SP
    df["trajectory"] = t
    return df

In [ ]:
if __name__ == "__main__":
    files = [
        "F0_SV0_SP1_t1_D1_R1.mat",
        "F2_SV2_SP2_t2.mat",
        "F3_SV3_SP1_t3.mat"
    ]

    all_df = []
    for f in files:
        df = process_file(f)
        all_df.append(df)

    combined = pd.concat(all_df, ignore_index=True)
    combined.to_csv("DronePropA_Full_Extraction.csv", index=False)
    print("Extraction complete → Saved as DronePropA_Full_Extraction.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'F0_SV0_SP1_t1_D1_R1.mat'

In [ ]:
data = pd.read_csv('DronePropA_Full_Extraction.csv')

# Data Exploration and Preparation

In [ ]:
data.head(10)

In [ ]:
data.isna().sum()

In [ ]:
data.shape[0]

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data = data.interpolate().fillna(method='bfill').fillna(method='ffill')

In [ ]:
data.isna().sum()

In [ ]:
data.shape

# Feature Engineering

**Objective:** Transform raw drone signals into structured, interpretable metrics for analysis and fault
 detection.

## Position and Orientation Errors</br>



*   Compute deviation between measured and reference positions (X, Y, Z).
*   Compute yaw
 misalignment using wrapped angular difference.





In [ ]:
data["error_x"] = data["meas_x"] - data["ref_x"]
data["error_y"] = data["meas_y"] - data["ref_y"]
data["error_z"] = data["meas_z"] - data["ref_z"]
data["pos_error_norm"] = np.sqrt(data["error_x"]**2 + data["error_y"]**2 + data["error_z"]**2)

# Safe yaw difference (handles wrap-around)
data["yaw_error"] = np.arctan2(np.sin(data["meas_yaw"] - data["ref_yaw"]),
                               np.cos(data["meas_yaw"] - data["ref_yaw"]))

## Velocity and Acceleration



*   Derive drone velocity and acceleration by differentiating position over time.
*   Compute overall
 speed magnitude to capture motion intensity.



In [ ]:
data["dt"] = data["time"].diff().fillna(method="bfill")

# Velocity
data["vel_x"] = data["meas_x"].diff() / data["dt"]
data["vel_y"] = data["meas_y"].diff() / data["dt"]
data["vel_z"] = data["meas_z"].diff() / data["dt"]
data["speed"] = np.sqrt(data["vel_x"]**2 + data["vel_y"]**2 + data["vel_z"]**2)

# Acceleration
data["acc_x"] = data["vel_x"].diff() / data["dt"]
data["acc_y"] = data["vel_y"].diff() / data["dt"]
data["acc_z"] = data["vel_z"].diff() / data["dt"]
data["acc_mag"] = np.sqrt(data["acc_x"]**2 + data["acc_y"]**2 + data["acc_z"]**2)

In [ ]:
data.head(20)

## Stability & Vibration Features


*   Use IMU signals (angular rates, accelerations) to quantify vibration levels.
*   High-frequency
 variations indicate propeller imbalance or structural fault.



In [ ]:
data["ang_vel_imu1"] = np.sqrt(
    data["imu1_roll_rate"]**2 + data["imu1_pitch_rate"]**2 + data["imu1_yaw_rate"]**2
)
data["ang_acc_imu1"] = np.sqrt(
    data["imu1_roll_acc"]**2 + data["imu1_pitch_acc"]**2 + data["imu1_yaw_acc"]**2
)
data["lin_acc_imu1"] = np.sqrt(
    data["accel1_x"]**2 + data["accel1_y"]**2 + data["accel1_z"]**2
)

## Actuator (Motor) Symmetry Features


*   Analyze motor command signals to detect imbalance between motors.
*   Compute mean, standard
 deviation, and asymmetry index across all four motors.



In [ ]:
motors = [
    "front_left_motor_cmd", "front_right_motor_cmd",
    "back_left_motor_cmd", "back_right_motor_cmd"
]

data["motor_mean"] = data[motors].mean(axis=1)
data["motor_std"] = data[motors].std(axis=1)
data["motor_asym"] = data["motor_std"] / (data["motor_mean"] + 1e-6)

## Energy / Power Proxy


*   Measure fluctuations in reference thrust as an indicator of control effort.
*    Smoother thrust profile implies stable dynamics.



In [ ]:
data["thrust_fluctuation"] = data["ref_thrust"].diff().abs()
data["thrust_smoothness"] = data["thrust_fluctuation"].rolling(10).mean()

## Statistical & Frequency Features


*   Aggregate per-flight metrics (mean, std, max, min) across each feature group.
*    Useful for ML model input representation.



In [ ]:
feature_cols = [
    "pos_error_norm", "yaw_error", "speed", "acc_mag",
    "ang_vel_imu1", "ang_acc_imu1", "lin_acc_imu1",
    "motor_asym", "motor_mean", "thrust_smoothness"
]

summary = data.groupby(["fault_type", "severity", "trajectory"]).agg({
    col: ["mean", "std", "max", "min"] for col in feature_cols
})
summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
summary = summary.reset_index()
summary.to_csv("DronePropA_Feature_Summary.csv", index=False)


NameError: name 'data' is not defined

## Feature Correlation Check


*   Compute correlation matrix across engineered features.
*   Identify redundant or highly correlated
 metrics before model training.



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = data[feature_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", annot=True)
plt.title("Feature Correlation Heatmap")
plt.show()

## Engineered Data
 Store final engineered features as DronePropA_FeatureEngineered.csv for modeling and
 analysis.

In [ ]:
data.to_csv("DronePropA_FeatureEngineered.csv", index=False)
print("Feature engineering complete → Saved as DronePropA_FeatureEngineered.csv")

# TEST

Working on the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DronePropA_Full_Extraction.csv")

In [ ]:
df.head()

,time,meas_x,meas_y,meas_z,meas_yaw,ref_x,ref_y,ref_z,ref_yaw,ref_thrust,...,front_right_esc_cmd,back_left_motor_cmd,back_left_esc_cmd,back_right_motor_cmd,back_right_esc_cmd,flight_mode,fault_type,severity,speed_level,trajectory
0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.006299,0.0,0.006299,0.0,0.0,0,0,1,1
1,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.002722,...,0.0,0.006303,0.0,0.006303,0.0,0.0,0,0,1,1
2,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.005098,...,0.0,0.006289,0.0,0.006289,0.0,0.0,0,0,1,1
3,0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.007598,...,0.0,0.006303,0.0,0.006303,0.0,0.0,0,0,1,1
4,0.004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.009820,...,0.0,0.006303,0.0,0.006303,0.0,0.0,0,0,1,1


In [ ]:
flying_cols = [col for col in df.columns if col not in ['fault_type', 'severity', 'trajectory', 'speed_level']]

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
df[flying_cols] = scaler.fit_transform(df[flying_cols])

In [ ]:
df.head()

,time,meas_x,meas_y,meas_z,meas_yaw,ref_x,ref_y,ref_z,ref_yaw,ref_thrust,...,front_right_esc_cmd,back_left_motor_cmd,back_left_esc_cmd,back_right_motor_cmd,back_right_esc_cmd,flight_mode,fault_type,severity,speed_level,trajectory
0,-1.727591,0.24516,0.134198,-2.075371,0.142733,0.242647,0.123348,-1.985012,-0.261811,1.777226,...,-4.307124,-3.497791,-4.52724,-3.428961,-4.233214,0.0,0,0,1,1
1,-1.727552,0.24516,0.134198,-2.075371,0.142733,0.242647,0.123348,-1.985012,-0.261811,1.774186,...,-4.307124,-3.497766,-4.52724,-3.428937,-4.233214,0.0,0,0,1,1
2,-1.727512,0.24516,0.134198,-2.075371,0.142733,0.242647,0.123348,-1.985012,-0.261811,1.771533,...,-4.307124,-3.497868,-4.52724,-3.429035,-4.233214,0.0,0,0,1,1
3,-1.727472,0.24516,0.134198,-2.075371,0.142733,0.242647,0.123348,-1.985012,-0.261811,1.768740,...,-4.307124,-3.497766,-4.52724,-3.428937,-4.233214,0.0,0,0,1,1
4,-1.727432,0.24516,0.134198,-2.075371,0.142733,0.242647,0.123348,-1.985012,-0.261811,1.766259,...,-4.307124,-3.497766,-4.52724,-3.428937,-4.233214,0.0,0,0,1,1


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df = df.drop('flight_mode')

corr = df.corr()
plt.figure(figsize=(20, 20))
sns.heatmap(corr, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

KeyError: "['flight_mode'] not found in axis"

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260125 entries, 0 to 260124
Data columns (total 58 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   time                   260125 non-null  float64
 1   meas_x                 260125 non-null  float64
 2   meas_y                 260125 non-null  float64
 3   meas_z                 260125 non-null  float64
 4   meas_yaw               260125 non-null  float64
 5   ref_x                  260125 non-null  float64
 6   ref_y                  260125 non-null  float64
 7   ref_z                  260125 non-null  float64
 8   ref_yaw                260125 non-null  float64
 9   ref_thrust             260125 non-null  float64
 10  ref_roll               260125 non-null  float64
 11  ref_pitch              260125 non-null  float64
 12  ref_yaw_rate           260125 non-null  float64
 13  imu1_roll              260125 non-null  float64
 14  imu1_pitch             260125 non-nu

In [ ]:
n_classes = df['fault_type'].nunique()
n_classes

3

In [ ]:
df[flying_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
time,260125.0,-4.755071e-16,1.000002,-1.727591,-0.864924,-0.002256,0.860412,1.848837
meas_x,260125.0,0.000000e+00,1.000002,-1.487114,-1.103895,0.233924,0.274820,2.007141
meas_y,260125.0,-8.740939e-18,1.000002,-1.627620,-0.614987,0.112043,0.164514,1.861994
meas_z,260125.0,8.050405e-16,1.000002,-2.075371,-0.625574,0.638883,0.668570,0.924847
meas_yaw,260125.0,3.496376e-18,1.000002,-8.869359,-0.423578,-0.183997,0.086064,3.302349
ref_x,260125.0,9.702442e-17,1.000002,-1.357144,-1.357144,0.242647,0.249785,1.842439
ref_y,260125.0,-1.756929e-16,1.000002,-1.447836,-0.671533,0.123348,0.138317,1.694532
ref_z,260125.0,3.968386e-16,1.000002,-1.985012,-0.567447,0.672922,0.672922,0.850117
ref_yaw,260125.0,1.608333e-16,1.000002,-1.115017,-0.261811,-0.261811,-0.261811,3.205740
ref_thrust,260125.0,1.503442e-16,1.000002,-12.761618,-0.165864,0.063183,0.285735,20.126614


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(['fault_type', 'severity', 'speed_level', 'trajectory'], axis=1)
y = df[['fault_type', 'severity', 'speed_level', 'trajectory']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (208100, 54)
Shape of X_test: (52025, 54)
Shape of y_train: (208100, 4)
Shape of y_test: (52025, 4)


,fault_type,severity,speed_level,trajectory
0,0,0,1,1
1,0,0,1,1
2,0,0,1,1
3,0,0,1,1
4,0,0,1,1


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

pred = model.predict(X_test)


In [ ]:
pred[0]

array([3, 3, 1, 3])

In [ ]:
y_test.values[0]

array([3, 3, 1, 3])

In [ ]:
from sklearn.metrics import precision_score
import numpy as np

target_cols = ['fault_type', 'severity', 'speed_level', 'trajectory']

precisions = []
for i, col in enumerate(target_cols):
    p = precision_score(y_test.iloc[:, i], pred[:, i], average='macro')
    precisions.append(p)
    print(f"Precision for {col}: {p:.3f}")

print(f"\nAverage precision across outputs: {np.mean(precisions):.3f}")


Precision for fault_type: 1.000
Precision for severity: 1.000
Precision for speed_level: 1.000
Precision for trajectory: 1.000

Average precision across outputs: 1.000


In [ ]:
f = "/content/drive/MyDrive/F2_SV2_SP1_t4.mat"
new_data = process_file(f)

In [ ]:
new_data.head()

,time,meas_x,meas_y,meas_z,meas_yaw,ref_x,ref_y,ref_z,ref_yaw,ref_thrust,...,front_right_esc_cmd,back_left_motor_cmd,back_left_esc_cmd,back_right_motor_cmd,back_right_esc_cmd,flight_mode,fault_type,severity,speed_level,trajectory
0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.047358,0.006712,0.0,0.006712,0.0,0.0,2,2,1,4
1,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.001649,...,0.047358,0.006716,0.0,0.006716,0.0,0.0,2,2,1,4
2,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.004163,...,0.047358,0.006704,0.0,0.006704,0.0,0.0,2,2,1,4
3,0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.006399,...,0.047358,0.006716,0.0,0.006716,0.0,0.0,2,2,1,4
4,0.004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.008961,...,0.047358,0.006708,0.0,0.006708,0.0,0.0,2,2,1,4


In [ ]:
new_X_test = new_data.drop(['fault_type', 'severity', 'speed_level', 'trajectory'], axis=1)
new_y_test = new_data[['fault_type', 'severity', 'speed_level', 'trajectory']]

In [ ]:
new_X_test.head()

,time,meas_x,meas_y,meas_z,meas_yaw,ref_x,ref_y,ref_z,ref_yaw,ref_thrust,...,height_range,front_left_motor_cmd,front_left_esc_cmd,front_right_motor_cmd,front_right_esc_cmd,back_left_motor_cmd,back_left_esc_cmd,back_right_motor_cmd,back_right_esc_cmd,flight_mode
0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.006712,0.0,0.006712,0.047358,0.006712,0.0,0.006712,0.0,0.0
1,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.001649,...,0.0,0.006716,0.0,0.006716,0.047358,0.006716,0.0,0.006716,0.0,0.0
2,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.004163,...,0.0,0.006704,0.0,0.006704,0.047358,0.006704,0.0,0.006704,0.0,0.0
3,0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.006399,...,0.0,0.006716,0.0,0.006716,0.047358,0.006716,0.0,0.006716,0.0,0.0
4,0.004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.008961,...,0.0,0.006708,0.0,0.006708,0.047358,0.006708,0.0,0.006708,0.0,0.0


In [ ]:
new_y_test.head()

,fault_type,severity,speed_level,trajectory
0,2,2,1,4
1,2,2,1,4
2,2,2,1,4
3,2,2,1,4
4,2,2,1,4


In [ ]:
new_pred = model.predict(new_X_test)

In [ ]:
target_cols = ['fault_type', 'severity', 'speed_level', 'trajectory']

precisions = []
for i, col in enumerate(target_cols):
    p = precision_score(new_y_test.iloc[:, i], new_pred[:, i], average='macro')
    precisions.append(p)
    print(f"Precision for {col}: {p:.3f}")

print(f"\nAverage precision across outputs: {np.mean(precisions):.3f}")


Precision for fault_type: 0.000
Precision for severity: 0.000
Precision for speed_level: 1.000
Precision for trajectory: 0.000

Average precision across outputs: 0.250


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print(y.value_counts())

fault_type  severity  speed_level  trajectory
3           3         1            3             89869
0           0         1            1             87837
2           2         2            2             82419
Name: count, dtype: int64
